# Small MoE LLM — train on Colab GPU

Runs the full pipeline (data prep → train → evaluate → ablations) on a Colab GPU in **bfloat16**.

**Before running:** set the runtime to GPU — *Runtime → Change runtime type → T4 GPU (or A100)*.

Fill in `REPO_URL` below with your GitHub repo, then Run All.

In [ ]:
# 0. Check the GPU
!nvidia-smi

In [ ]:
# 1. Clone the repo (edit REPO_URL to your fork)
REPO_URL = 'https://github.com/<YOUR_USERNAME>/Small_MoE_LLM.git'
!git clone $REPO_URL
%cd Small_MoE_LLM

In [ ]:
# 2. Install deps. Colab already ships a CUDA build of torch — do NOT reinstall it
#    (that could downgrade to CPU). Install everything else.
!grep -v '^torch' requirements.txt > /tmp/reqs_colab.txt
!pip install -q -r /tmp/reqs_colab.txt
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported())

In [ ]:
# 3. Sanity: run the test suite (offline/synthetic parts) + a tiny smoke train
!python -m pytest -q tests/test_model_build.py tests/test_trainer.py
!python scripts/train.py --config configs/smoke.yaml

In [ ]:
# 4. Prepare real data (streams a capped subsample of each modality)
!python scripts/prepare_data.py --config configs/train_small.yaml --set data.max_samples_per_modality=20000

In [ ]:
# 5. Train the FULL model (172M, bf16, real data). Adjust max_steps to your time budget.
#    On a T4 keep batch small; on A100 you can raise it.
!python scripts/train.py --config configs/train_small.yaml \
    --set training.max_steps=3000 training.per_device_batch_size=8 logging.backend=tensorboard

In [ ]:
# 6. Watch training curves in TensorBoard
%load_ext tensorboard
%tensorboard --logdir checkpoints/small-moe-baseline/tb

In [ ]:
# 7. Evaluate the trained checkpoint (metrics + routing heatmap)
!python scripts/evaluate.py --config configs/train_small.yaml \
    --checkpoint checkpoints/small-moe-baseline/final --n_examples 200
from IPython.display import Image
Image('checkpoints/small-moe-baseline/final/routing_heatmap.png')

In [ ]:
# 8. (optional) Run the ablation matrix on GPU
!python scripts/run_ablations.py --steps 500 --out report/figures
from IPython.display import Image
Image('report/figures/ablation_comparison.png')

In [ ]:
# 9. Plot training metrics + download results
!python scripts/plot_metrics.py --log logs/metrics.jsonl --out report/figures
!zip -r results.zip report/figures checkpoints/small-moe-baseline/final/results.json \
    checkpoints/small-moe-baseline/final/summary.md logs
from google.colab import files
files.download('results.zip')